M

# Notebook 03: Data Enrichment — Gender via Wikidata

**Project:** Who Makes the Canon? Gender Representation in European Painting Collections  
**Author:** Samiulhaq  
**Date:** June 2026

## Purpose
Match artist names to Wikidata to extract gender, birth year, and nationality.

## Input
- `data/processed/europeana_clean.csv`

## Output
- `data/processed/europeana_enriched.csv`
- `data/processed/gender_enrichment_log.csv`

In [1]:
# ============================================================
# IMPORTS AND CONFIGURATION
# ============================================================

import pandas as pd
import requests
import time
import os

INPUT_FILE      = r"C:\Users\bfde3\OneDrive\Documents\europeana-gender-painting\data\processed\europeana_clean.csv"
OUTPUT_FILE     = r"C:\Users\bfde3\OneDrive\Documents\europeana-gender-painting\data\processed\europeana_enriched.csv"
ENRICHMENT_LOG  = r"C:\Users\bfde3\OneDrive\Documents\europeana-gender-painting\data\processed\gender_enrichment_log.csv"

df = pd.read_csv(INPUT_FILE)
print(f"Loaded clean data: {df.shape[0]:,} rows")

# Get unique named artists only
named_artists = df[df["creator_type"] == "named"]["creator"].dropna().unique()
print(f"Unique named artists to look up: {len(named_artists):,}")

Loaded clean data: 27,454 rows
Unique named artists to look up: 8,837


In [2]:
# ============================================================
# WIKIDATA FUNCTIONS
# ============================================================

WIKIDATA_URL = "https://www.wikidata.org/w/api.php"

def get_wikidata_properties(q_id):
    props = {"gender": "unknown", "birth_year": None, "nationality": None}
    params = {
        "action": "wbgetentities",
        "ids": q_id,
        "props": "claims",
        "format": "json"
    }
    try:
        r = requests.get(WIKIDATA_URL, params=params, timeout=15)
        entity = r.json().get("entities", {}).get(q_id, {})
        claims = entity.get("claims", {})
        
        # P21 = gender
        if "P21" in claims:
            gender_id = claims["P21"][0]["mainsnak"]["datavalue"]["value"]["id"]
            if gender_id == "Q6581072":
                props["gender"] = "female"
            elif gender_id == "Q6581097":
                props["gender"] = "male"
        
        # P569 = birth date
        if "P569" in claims:
            try:
                birth_str = claims["P569"][0]["mainsnak"]["datavalue"]["value"]["time"]
                props["birth_year"] = int(birth_str[1:5])
            except:
                pass
                
        # P27 = nationality
        if "P27" in claims:
            props["nationality"] = claims["P27"][0]["mainsnak"]["datavalue"]["value"]["id"]
    except:
        pass
    return props


def search_wikidata_artist(name):
    result = {
        "creator": name,
        "wikidata_id": None,
        "gender": "unknown",
        "birth_year": None,
        "nationality": None,
        "match_status": "not_found"
    }
    
    search_params = {
        "action": "wbsearchentities",
        "search": name,
        "language": "en",
        "type": "item",
        "limit": 5,
        "format": "json"
    }
    
    try:
        r = requests.get(WIKIDATA_URL, params=search_params, timeout=15)
        candidates = r.json().get("search", [])
        
        if not candidates:
            return result
        
        art_keywords = ["painter", "artist", "sculptor", "engraver",
                       "printmaker", "illustrator", "draughtsman"]
        
        for candidate in candidates:
            q_id = candidate.get("id")
            description = candidate.get("description", "").lower()
            is_artist = any(kw in description for kw in art_keywords)
            
            if is_artist or candidates.index(candidate) == 0:
                props = get_wikidata_properties(q_id)
                result["wikidata_id"] = q_id
                result["gender"] = props["gender"]
                result["birth_year"] = props["birth_year"]
                result["nationality"] = props["nationality"]
                result["match_status"] = "matched" if is_artist else "uncertain"
                break
    except Exception as e:
        result["match_status"] = "error"
    
    return result

print("Wikidata functions ready!")

Wikidata functions ready!


In [3]:
# ============================================================
# STEP 1: Test with 10 artists first
# ============================================================

print("Testing with 10 artists first...\n")

test_results = []
for i, artist in enumerate(named_artists[:10]):
    print(f"  [{i+1}/10] Looking up: {artist}")
    result = search_wikidata_artist(artist)
    test_results.append(result)
    time.sleep(0.5)

df_test = pd.DataFrame(test_results)
print(f"\nTest results:")
print(df_test[["creator", "gender", "birth_year", "match_status"]])

Testing with 10 artists first...

  [1/10] Looking up: Bonnet, Sylvain (1645? -1705). Painter
  [2/10] Looking up: Rembrandt (1606-1669). Graveur
  [3/10] Looking up: Audran, Gérard (1640-1703). Graveur
  [4/10] Looking up: Oppenort, Gilles-Marie (1685-1742). Dessinateur
  [5/10] Looking up: http://viaf.org/viaf/24606800
  [6/10] Looking up: Picart, Étienne (1632-1721). Graveur
  [7/10] Looking up: Corrège, Le (1489? -1534). Model Painter
  [8/10] Looking up: Cotte, Robert de (1656-1735). Dessinateur
  [9/10] Looking up: Boudan, Louis (16. -17; designer and engraver). Drawer
  [10/10] Looking up: Leclerc, Sébastien (1637-1714). Graveur

Test results:
                                             creator   gender birth_year  \
0             Bonnet, Sylvain (1645? -1705). Painter  unknown       None   
1                     Rembrandt (1606-1669). Graveur  unknown       None   
2                Audran, Gérard (1640-1703). Graveur  unknown       None   
3    Oppenort, Gilles-Marie (1685-174

In [4]:
# ============================================================
# FIX: Clean artist names before Wikidata search
# Remove dates, job titles, and extra information
# Example: "Rembrandt (1606-1669). Graveur" → "Rembrandt"
# ============================================================

import re

def clean_artist_name(name):
    """
    Extract just the person's name from messy creator strings.
    Removes dates, job titles, URLs, and extra info.
    """
    if pd.isna(name):
        return None
    
    name = str(name)
    
    # Skip URLs entirely
    if name.startswith("http"):
        return None
    
    # Remove anything in parentheses (dates like "(1606-1669)")
    name = re.sub(r'\(.*?\)', '', name)
    
    # Remove job titles after a period or comma
    # e.g. "Rembrandt. Graveur" → "Rembrandt"
    # e.g. "Bonnet, Sylvain. Painter" → "Bonnet, Sylvain"
    name = re.sub(r'\.\s*(Painter|Graveur|Dessinateur|Drawer|Engraver|Illustrateur|Peintre|Sculptor|Printmaker|Artist|Designer|Photographer).*$', '', name, flags=re.IGNORECASE)
    
    # Remove extra whitespace
    name = ' '.join(name.split())
    name = name.strip().strip(',').strip()
    
    # Skip if too short or empty
    if len(name) < 3:
        return None
    
    return name

# Test the cleaning function
test_names = [
    "Bonnet, Sylvain (1645? -1705). Painter",
    "Rembrandt (1606-1669). Graveur",
    "Audran, Gérard (1640-1703). Graveur",
    "http://viaf.org/viaf/24606800",
    "Picart, Étienne (1632-1721). Graveur",
]

print("Name cleaning test:")
print(f"{'Original':<50} {'Cleaned':<30}")
print("-" * 80)
for name in test_names:
    cleaned = clean_artist_name(name)
    print(f"{name:<50} {str(cleaned):<30}")

Name cleaning test:
Original                                           Cleaned                       
--------------------------------------------------------------------------------
Bonnet, Sylvain (1645? -1705). Painter             Bonnet, Sylvain               
Rembrandt (1606-1669). Graveur                     Rembrandt                     
Audran, Gérard (1640-1703). Graveur                Audran, Gérard                
http://viaf.org/viaf/24606800                      None                          
Picart, Étienne (1632-1721). Graveur               Picart, Étienne               


In [5]:
# ============================================================
# STEP 2: Run enrichment with cleaned names
# ============================================================

# Add cleaned name column to main dataset
df["creator_clean"] = df["creator"].apply(clean_artist_name)

# Get unique cleaned names (skip None values)
unique_clean = df[df["creator_clean"].notna()]["creator_clean"].unique()
print(f"Unique cleaned artist names: {len(unique_clean):,}")

# Run enrichment
print(f"\nStarting Wikidata enrichment...")
print(f"This will take approximately 60-90 minutes")
print(f"Do NOT close Jupyter while running!\n")

enrichment_results = []

for i, artist_name in enumerate(unique_clean):
    
    # Progress update every 100 artists
    if i % 100 == 0:
        print(f"  Progress: {i:,}/{len(unique_clean):,} artists processed...")
    
    result = search_wikidata_artist(artist_name)
    result["creator_clean"] = artist_name
    enrichment_results.append(result)
    
    time.sleep(0.3)

print(f"\nEnrichment complete!")
df_enrichment = pd.DataFrame(enrichment_results)

print(f"\nMatch status breakdown:")
print(df_enrichment["match_status"].value_counts())
print(f"\nGender breakdown:")
print(df_enrichment["gender"].value_counts())

# Save enrichment log
df_enrichment.to_csv(ENRICHMENT_LOG, index=False, encoding="utf-8")
print(f"\nEnrichment log saved!")

Unique cleaned artist names: 7,389

Starting Wikidata enrichment...
This will take approximately 60-90 minutes
Do NOT close Jupyter while running!

  Progress: 0/7,389 artists processed...
  Progress: 100/7,389 artists processed...
  Progress: 200/7,389 artists processed...
  Progress: 300/7,389 artists processed...
  Progress: 400/7,389 artists processed...
  Progress: 500/7,389 artists processed...
  Progress: 600/7,389 artists processed...
  Progress: 700/7,389 artists processed...
  Progress: 800/7,389 artists processed...
  Progress: 900/7,389 artists processed...
  Progress: 1,000/7,389 artists processed...
  Progress: 1,100/7,389 artists processed...
  Progress: 1,200/7,389 artists processed...
  Progress: 1,300/7,389 artists processed...
  Progress: 1,400/7,389 artists processed...
  Progress: 1,500/7,389 artists processed...
  Progress: 1,600/7,389 artists processed...
  Progress: 1,700/7,389 artists processed...
  Progress: 1,800/7,389 artists processed...
  Progress: 1,900/7

In [6]:
# ============================================================
# FIX: Test Wikidata connection
# ============================================================

import requests

try:
    r = requests.get(
        "https://www.wikidata.org/w/api.php",
        params={
            "action": "wbsearchentities",
            "search": "Rembrandt",
            "language": "en",
            "type": "item",
            "limit": 3,
            "format": "json"
        },
        timeout=15
    )
    print(f"Status: {r.status_code}")
    print(f"Response: {r.json()}")
except Exception as e:
    print(f"Failed: {e}")

Status: 403
Failed: Expecting value: line 1 column 1 (char 0)


In [7]:
# ============================================================
# ALTERNATIVE APPROACH: Gender dictionary of known artists
# This is more reliable than API calls
# Based on well-documented artists in European collections
# ============================================================

# Dictionary of cleaned artist names → gender
# female = confirmed female artist
# male = confirmed male artist
KNOWN_GENDERS = {
    # Female artists
    "morisot, berthe": "female",
    "cassatt, mary": "female",
    "valadon, suzanne": "female",
    "bashkirtseff, marie": "female",
    "vigee le brun, elisabeth": "female",
    "vigée le brun, élisabeth": "female",
    "anguissola, sofonisba": "female",
    "gentileschi, artemisia": "female",
    "peeters, clara": "female",
    "fontana, lavinia": "female",
    "kauffmann, angelica": "female",
    "kauffman, angelica": "female",
    "hepworth, barbara": "female",
    "kollwitz, kathe": "female",
    "kollwitz, käthe": "female",
    "münter, gabriele": "female",
    "munter, gabriele": "female",
    "goncharova, natalia": "female",
    "gonzales, eva": "female",
    "bonheur, rosa": "female",
    "neel, alice": "female",
    "popova, liubov": "female",
    "stepanova, varvara": "female",
    "hilma af klint": "female",
    "af klint, hilma": "female",
    "modersohn-becker, paula": "female",
    "modersohn becker, paula": "female",
    "brooks, romaine": "female",
    "laurencin, marie": "female",
    "delaunay, sonia": "female",
    "tanning, dorothea": "female",
    "carrington, leonora": "female",
    "oppenheim, meret": "female",
    "hoch, hannah": "female",
    "höch, hannah": "female",
    "leyster, judith": "female",
    "ruysch, rachel": "female",
    "oosterwyck, maria van": "female",
    "merian, maria sibylla": "female",
    "lama, giulia": "female",
    "lisiewska, anna dorothea": "female",
    "therbusch, anna dorothea": "female",
    "labille-guiard, adelaide": "female",
    "labille-guiard, adélaïde": "female",
    "moillon, louise": "female",
    "vallayer-coster, anne": "female",
    "cameron, julia margaret": "female",
    "butler, lady elizabeth": "female",
    "osborn, emily mary": "female",
    "siddal, elizabeth": "female",
    "spartali stillman, marie": "female",
    "bracquemond, marie": "female",
    "potter, paulus": "male",

    # Male artists — major ones
    "rembrandt": "male",
    "rembrandt van rijn": "male",
    "picasso, pablo": "male",
    "picasso": "male",
    "monet, claude": "male",
    "renoir, pierre-auguste": "male",
    "degas, edgar": "male",
    "cezanne, paul": "male",
    "cézanne, paul": "male",
    "manet, edouard": "male",
    "manet, édouard": "male",
    "van gogh, vincent": "male",
    "gogh, vincent van": "male",
    "rubens, peter paul": "male",
    "raphael": "male",
    "michelangelo": "male",
    "leonardo da vinci": "male",
    "caravaggio": "male",
    "velazquez, diego": "male",
    "velázquez, diego": "male",
    "goya, francisco": "male",
    "goya, francisco de": "male",
    "turner, j.m.w.": "male",
    "constable, john": "male",
    "delacroix, eugene": "male",
    "delacroix, eugène": "male",
    "courbet, gustave": "male",
    "millet, jean-francois": "male",
    "millet, jean-françois": "male",
    "corot, jean-baptiste-camille": "male",
    "daumier, honore": "male",
    "daumier, honoré": "male",
    "ingres, jean-auguste-dominique": "male",
    "david, jacques-louis": "male",
    "gericault, theodore": "male",
    "géricault, théodore": "male",
    "klimt, gustav": "male",
    "schiele, egon": "male",
    "munch, edvard": "male",
    "kandinsky, wassily": "male",
    "klee, paul": "male",
    "mondrian, piet": "male",
    "matisse, henri": "male",
    "chagall, marc": "male",
    "dali, salvador": "male",
    "dalí, salvador": "male",
    "ernst, max": "male",
    "magritte, rene": "male",
    "magritte, rené": "male",
    "dix, otto": "male",
    "grosz, george": "male",
    "beckmann, max": "male",
    "kirchner, ernst ludwig": "male",
    "nolde, emil": "male",
    "marc, franz": "male",
    "macke, august": "male",
    "jawlensky, alexej von": "male",
    "kokoschka, oskar": "male",
    "hodler, ferdinand": "male",
    "bocklin, arnold": "male",
    "böcklin, arnold": "male",
    "menzel, adolph von": "male",
    "leibl, wilhelm": "male",
    "liebermann, max": "male",
    "corinth, lovis": "male",
    "slevogt, max": "male",
    "zorn, anders": "male",
    "larsson, carl": "male",
    "kroyer, peder severin": "male",
    "krøyer, peder severin": "male",
    "hammershoi, vilhelm": "male",
    "hammershøi, vilhelm": "male",
    "eckersberg, christoffer wilhelm": "male",
    "marstrand, wilhelm": "male",
    "lundbye, johan thomas": "male",
    "skovgaard, peter christian": "male",
    "ancher, michael": "male",
    "ancher, anna": "female",
    "brendekilde, hans andersen": "male",
    "exner, julius": "male",
    "titian": "male",
    "tintoretto": "male",
    "veronese, paolo": "male",
    "tiepolo, giovanni battista": "male",
    "canaletto": "male",
    "bellini, giovanni": "male",
    "botticelli, sandro": "male",
    "mantegna, andrea": "male",
    "masaccio": "male",
    "fra angelico": "male",
    "piero della francesca": "male",
    "lippi, fra filippo": "male",
    "ghirlandaio, domenico": "male",
    "signorelli, luca": "male",
    "perugino": "male",
    "sodoma": "male",
    "pontormo": "male",
    "bronzino": "male",
    "vasari, giorgio": "male",
    "carracci, annibale": "male",
    "reni, guido": "male",
    "guercino": "male",
    "piranesi, giovanni battista": "male",
    "canova, antonio": "male",
    "ingres": "male",
    "boucher, francois": "male",
    "boucher, françois": "male",
    "fragonard, jean-honore": "male",
    "fragonard, jean-honoré": "male",
    "watteau, antoine": "male",
    "chardin, jean-baptiste-simeon": "male",
    "chardin, jean-baptiste-siméon": "male",
    "poussin, nicolas": "male",
    "claude lorrain": "male",
    "le brun, charles": "male",
    "rigaud, hyacinthe": "male",
    "nattier, jean-marc": "male",
    "hals, frans": "male",
    "vermeer, johannes": "male",
    "steen, jan": "male",
    "ruisdael, jacob van": "male",
    "hobbema, meindert": "male",
    "hooch, pieter de": "male",
    "ter borch, gerard": "male",
    "metsu, gabriel": "male",
    "wouwerman, philips": "male",
    "dou, gerrit": "male",
    "van dyck, anthony": "male",
    "dyck, anthony van": "male",
    "jordaens, jacob": "male",
    "brueghel, pieter": "male",
    "bruegel, pieter": "male",
    "bosch, hieronymus": "male",
    "memling, hans": "male",
    "van eyck, jan": "male",
    "eyck, jan van": "male",
    "rogier van der weyden": "male",
    "weyden, rogier van der": "male",
    "velazquez": "male",
    "el greco": "male",
    "zurbaran, francisco de": "male",
    "zurbarán, francisco de": "male",
    "murillo, bartolome esteban": "male",
    "murillo, bartolomé esteban": "male",
    "ribera, jose de": "male",
    "ribera, josé de": "male",
    "sorolla, joaquin": "male",
    "sorolla, joaquín": "male",
    "fortuny, mariano": "male",
    "gainsborough, thomas": "male",
    "reynolds, joshua": "male",
    "hogarth, william": "male",
    "blake, william": "male",
    "stubbs, george": "male",
    "romney, george": "male",
    "raeburn, henry": "male",
    "lawrence, thomas": "male",
    "landseer, edwin": "male",
    "millais, john everett": "male",
    "rossetti, dante gabriel": "male",
    "burne-jones, edward": "male",
    "hunt, william holman": "male",
    "leighton, frederic": "male",
    "alma-tadema, lawrence": "male",
    "sargent, john singer": "male",
    "whistler, james mcneill": "male",
    "ensor, james": "male",
    "spilliaert, leon": "male",
    "spilliaert, léon": "male",
    "permeke, constant": "male",
    "wouters, rik": "male",
    "de braekeleer, henri": "male",
    "leys, baron hendrik": "male",
    "gallait, louis": "male",
    "wiertz, antoine": "male",
    "courbet": "male",
}

def lookup_gender(clean_name):
    """Look up gender from our dictionary"""
    if pd.isna(clean_name) or clean_name is None:
        return "unknown"
    
    # Try exact match first (lowercase)
    key = str(clean_name).lower().strip()
    if key in KNOWN_GENDERS:
        return KNOWN_GENDERS[key]
    
    # Try partial match — check if any known name is contained in the artist name
    for known_name, gender in KNOWN_GENDERS.items():
        if known_name in key or key in known_name:
            return gender
    
    return "unknown"

# Test
test_names = ["Rembrandt", "Morisot, Berthe", "Cassatt, Mary", "Picasso, Pablo", "Unknown Artist"]
print("Gender lookup test:")
for name in test_names:
    print(f"  {name:<30} → {lookup_gender(name)}")

Gender lookup test:
  Rembrandt                      → male
  Morisot, Berthe                → female
  Cassatt, Mary                  → female
  Picasso, Pablo                 → male
  Unknown Artist                 → unknown


In [8]:
# ============================================================
# APPLY GENDER TO ALL RECORDS
# ============================================================

# Apply gender lookup to all records
df["gender"] = df["creator_clean"].apply(lookup_gender)

# Summary
print("GENDER ENRICHMENT RESULTS")
print("="*50)
print(f"\nGender breakdown:")
print(df["gender"].value_counts())
print(f"\nAs percentages:")
print(df["gender"].value_counts(normalize=True).mul(100).round(1))

# How many named artists got a gender assigned?
named_df = df[df["creator_type"] == "named"]
print(f"\nOf {len(named_df):,} named artists:")
print(f"  Gender identified: {(named_df['gender'] != 'unknown').sum():,}")
print(f"  Still unknown:     {(named_df['gender'] == 'unknown').sum():,}")

GENDER ENRICHMENT RESULTS

Gender breakdown:
gender
unknown    27258
male         170
female        26
Name: count, dtype: int64

As percentages:
gender
unknown    99.3
male        0.6
female      0.1
Name: proportion, dtype: float64

Of 26,360 named artists:
  Gender identified: 195
  Still unknown:     26,165


In [9]:
# ============================================================
# FIX: Use a different Wikidata endpoint that is not blocked
# We use the SPARQL endpoint instead
# ============================================================

import requests

# Test SPARQL endpoint
SPARQL_URL = "https://query.wikidata.org/sparql"

test_query = """
SELECT ?item ?itemLabel ?genderLabel ?birthYear WHERE {
  ?item wdt:P31 wd:Q5 .
  ?item wdt:P21 ?gender .
  ?item rdfs:label "Rembrandt"@en .
  OPTIONAL { ?item wdt:P569 ?birth . BIND(YEAR(?birth) AS ?birthYear) }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" . }
}
LIMIT 5
"""

headers = {
    "User-Agent": "EuropeanaGenderProject/1.0",
    "Accept": "application/json"
}

try:
    r = requests.get(
        SPARQL_URL,
        params={"query": test_query, "format": "json"},
        headers=headers,
        timeout=30
    )
    print(f"Status: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        results = data.get("results", {}).get("bindings", [])
        print(f"Results found: {len(results)}")
        for result in results:
            print(f"  {result}")
    else:
        print(f"Response: {r.text[:200]}")
except Exception as e:
    print(f"Failed: {e}")

Status: 200
Results found: 1
  {'item': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q5598'}, 'birthYear': {'datatype': 'http://www.w3.org/2001/XMLSchema#integer', 'type': 'literal', 'value': '1606'}, 'itemLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'Rembrandt'}, 'genderLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'male'}}


In [10]:
# ============================================================
# SPARQL-BASED GENDER ENRICHMENT
# This works because SPARQL endpoint is not blocked!
# ============================================================

SPARQL_URL = "https://query.wikidata.org/sparql"

headers = {
    "User-Agent": "EuropeanaGenderProject/1.0",
    "Accept": "application/json"
}

def get_gender_sparql(name):
    """
    Query Wikidata SPARQL for artist gender by name.
    Returns: 'male', 'female', or 'unknown'
    """
    # Clean the name for SPARQL query
    # Escape quotes
    safe_name = str(name).replace('"', '').replace("'", "")
    
    query = f"""
    SELECT ?genderLabel ?birthYear WHERE {{
      ?item wdt:P31 wd:Q5 .
      ?item wdt:P21 ?gender .
      ?item rdfs:label "{safe_name}"@en .
      OPTIONAL {{ 
        ?item wdt:P569 ?birth . 
        BIND(YEAR(?birth) AS ?birthYear) 
      }}
      SERVICE wikibase:label {{ 
        bd:serviceParam wikibase:language "en" . 
      }}
    }}
    LIMIT 1
    """
    
    try:
        r = requests.get(
            SPARQL_URL,
            params={{"query": query, "format": "json"}},
            headers=headers,
            timeout=15
        )
        
        if r.status_code != 200:
            return "unknown", None
            
        results = r.json().get("results", {}).get("bindings", [])
        
        if not results:
            return "unknown", None
            
        gender = results[0].get("genderLabel", {}).get("value", "unknown")
        birth_year = results[0].get("birthYear", {}).get("value", None)
        
        if gender not in ["male", "female"]:
            gender = "unknown"
            
        return gender, birth_year
        
    except Exception as e:
        return "unknown", None


# Test with known artists
test_artists = [
    "Rembrandt",
    "Berthe Morisot",
    "Mary Cassatt",
    "Pablo Picasso",
    "Artemisia Gentileschi",
    "Clara Peeters",
    "Gustav Klimt",
]

print("SPARQL gender lookup test:")
print(f"{'Artist':<30} {'Gender':<10} {'Birth Year'}")
print("-" * 55)

for artist in test_artists:
    gender, birth = get_gender_sparql(artist)
    print(f"{artist:<30} {gender:<10} {birth}")
    time.sleep(1)

SPARQL gender lookup test:
Artist                         Gender     Birth Year
-------------------------------------------------------
Rembrandt                      unknown    None
Berthe Morisot                 unknown    None
Mary Cassatt                   unknown    None
Pablo Picasso                  unknown    None
Artemisia Gentileschi          unknown    None
Clara Peeters                  unknown    None
Gustav Klimt                   unknown    None


In [11]:
# ============================================================
# FIX: Better SPARQL query with label search
# ============================================================

def get_gender_sparql(name):
    """
    Query Wikidata SPARQL for artist gender by name.
    Uses label service for better matching.
    """
    safe_name = str(name).replace('"', '').replace("'", "").strip()
    
    # Try first name format as-is, then reversed
    # e.g. "Morisot, Berthe" → try "Berthe Morisot" too
    name_variants = [safe_name]
    
    # If name has comma, try reversed version
    if "," in safe_name:
        parts = safe_name.split(",", 1)
        reversed_name = parts[1].strip() + " " + parts[0].strip()
        name_variants.append(reversed_name)
    
    for variant in name_variants:
        query = f"""
        SELECT ?item ?genderLabel ?birthYear WHERE {{
          VALUES ?name {{ "{variant}"@en "{variant}"@fr 
                         "{variant}"@de "{variant}"@nl }}
          ?item rdfs:label ?name .
          ?item wdt:P31 wd:Q5 .
          ?item wdt:P21 ?gender .
          OPTIONAL {{ 
            ?item wdt:P569 ?birth . 
            BIND(YEAR(?birth) AS ?birthYear) 
          }}
          SERVICE wikibase:label {{ 
            bd:serviceParam wikibase:language "en" . 
          }}
        }}
        LIMIT 1
        """
        
        try:
            r = requests.get(
                SPARQL_URL,
                params={"query": query, "format": "json"},
                headers=headers,
                timeout=20
            )
            
            if r.status_code != 200:
                continue
                
            results = r.json().get("results", {}).get("bindings", [])
            
            if results:
                gender = results[0].get("genderLabel", {}).get("value", "unknown")
                birth_year = results[0].get("birthYear", {}).get("value", None)
                if gender in ["male", "female"]:
                    return gender, birth_year
                    
        except Exception as e:
            continue
        
        time.sleep(0.5)
    
    return "unknown", None


# Test with known artists
test_artists = [
    "Rembrandt",
    "Morisot, Berthe",
    "Cassatt, Mary",
    "Picasso, Pablo",
    "Gentileschi, Artemisia",
    "Peeters, Clara",
    "Klimt, Gustav",
    "Kollwitz, Käthe",
]

print("SPARQL gender lookup test:")
print(f"{'Artist':<35} {'Gender':<10} {'Birth Year'}")
print("-" * 60)

for artist in test_artists:
    gender, birth = get_gender_sparql(artist)
    print(f"{artist:<35} {gender:<10} {birth}")
    time.sleep(1)

SPARQL gender lookup test:
Artist                              Gender     Birth Year
------------------------------------------------------------
Rembrandt                           male       1606
Morisot, Berthe                     female     1841
Cassatt, Mary                       female     1844
Picasso, Pablo                      male       1881
Gentileschi, Artemisia              female     1593
Peeters, Clara                      female     1587
Klimt, Gustav                       male       1862
Kollwitz, Käthe                     female     1867


In [12]:
# ============================================================
# FULL ENRICHMENT LOOP
# Process all unique cleaned artist names
# WARNING: Takes 2-3 hours - do NOT close Jupyter!
# ============================================================

print(f"Starting full enrichment for {len(unique_clean):,} artists...")
print("Do NOT close Jupyter while running!\n")

enrichment_results = []

for i, artist_name in enumerate(unique_clean):
    
    # Progress every 50 artists
    if i % 50 == 0:
        matched = sum(1 for r in enrichment_results 
                     if r.get("gender") in ["male", "female"])
        print(f"  [{i:,}/{len(unique_clean):,}] "
              f"Matched so far: {matched:,}")
    
    gender, birth_year = get_gender_sparql(artist_name)
    
    enrichment_results.append({
        "creator_clean":  artist_name,
        "gender":         gender,
        "birth_year":     birth_year,
        "match_status":   "matched" if gender != "unknown" else "not_found"
    })
    
    # Save progress every 500 artists in case of interruption
    if i % 500 == 0 and i > 0:
        df_temp = pd.DataFrame(enrichment_results)
        df_temp.to_csv(ENRICHMENT_LOG, index=False, encoding="utf-8")
        print(f"  Progress saved at {i:,} artists!")
    
    time.sleep(0.8)  # polite pause for SPARQL endpoint


# Final save
df_enrichment = pd.DataFrame(enrichment_results)
df_enrichment.to_csv(ENRICHMENT_LOG, index=False, encoding="utf-8")

print(f"\n{'='*50}")
print(f"ENRICHMENT COMPLETE!")
print(f"{'='*50}")
print(f"\nMatch status:")
print(df_enrichment["match_status"].value_counts())
print(f"\nGender breakdown:")
print(df_enrichment["gender"].value_counts())

Starting full enrichment for 7,389 artists...
Do NOT close Jupyter while running!

  [0/7,389] Matched so far: 0
  [50/7,389] Matched so far: 26
  [100/7,389] Matched so far: 46
  [150/7,389] Matched so far: 72
  [200/7,389] Matched so far: 87
  [250/7,389] Matched so far: 98
  [300/7,389] Matched so far: 112
  [350/7,389] Matched so far: 123
  [400/7,389] Matched so far: 136
  [450/7,389] Matched so far: 142
  [500/7,389] Matched so far: 146
  Progress saved at 500 artists!
  [550/7,389] Matched so far: 150
  [600/7,389] Matched so far: 160
  [650/7,389] Matched so far: 162
  [700/7,389] Matched so far: 169
  [750/7,389] Matched so far: 177
  [800/7,389] Matched so far: 184
  [850/7,389] Matched so far: 202
  [900/7,389] Matched so far: 225
  [950/7,389] Matched so far: 260
  [1,000/7,389] Matched so far: 277
  Progress saved at 1,000 artists!
  [1,050/7,389] Matched so far: 278
  [1,100/7,389] Matched so far: 278
  [1,150/7,389] Matched so far: 278
  [1,200/7,389] Matched so far: 287

In [13]:
# ============================================================
# MERGE GENDER INTO MAIN DATASET AND SAVE
# ============================================================

# Merge enrichment results back into main dataframe
df_enrichment = pd.DataFrame(enrichment_results)

df_final = df.merge(
    df_enrichment[["creator_clean", "gender", "birth_year", "match_status"]],
    on="creator_clean",
    how="left",
    suffixes=("_old", "")
)

# Drop old placeholder columns
cols_to_drop = [c for c in df_final.columns if c.endswith("_old")]
df_final = df_final.drop(columns=cols_to_drop)

# For anonymous records set gender to "anonymous"
df_final.loc[df_final["creator_type"] == "anonymous", "gender"] = "anonymous"

print("FINAL DATASET SUMMARY")
print("="*50)
print(f"Total records: {len(df_final):,}")
print(f"\nGender breakdown:")
print(df_final["gender"].value_counts())
print(f"\nAs percentages:")
print(df_final["gender"].value_counts(normalize=True).mul(100).round(1))

# Save enriched dataset
df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print(f"\nEnriched dataset saved to:")
print(OUTPUT_FILE)

if os.path.exists(OUTPUT_FILE):
    size = os.path.getsize(OUTPUT_FILE) / 1024 / 1024
    print(f"File size: {size:.1f} MB")
    print("\nNotebook 03 complete!")

FINAL DATASET SUMMARY
Total records: 27,454

Gender breakdown:
gender
unknown      12883
male          7977
anonymous     1094
female         713
Name: count, dtype: int64

As percentages:
gender
unknown      56.8
male         35.2
anonymous     4.8
female        3.1
Name: proportion, dtype: float64

Enriched dataset saved to:
C:\Users\bfde3\OneDrive\Documents\europeana-gender-painting\data\processed\europeana_enriched.csv
File size: 17.4 MB

Notebook 03 complete!
